### 출력파서

LLM이 생성 출력 결과를 필요에 맞게 가공, 구조화

In [49]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [50]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [51]:
output_parser

StrOutputParser()

In [52]:
output_parser.invoke("테스트") # runnable

'테스트'

In [53]:
from langchain_core.prompts import PromptTemplate

template = "{language} 할 수 있어?"

prompt = PromptTemplate.from_template(template)

In [54]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0, 
    google_api_key=gemini_api_key
)

In [55]:
chain = prompt | llm | output_parser # 체인 구성

In [56]:
chain.invoke({"language" : "한국어"})

'네, 한국어 할 수 있습니다. 어떤 질문이든 한국어로 답변해 드릴 수 있습니다!'

### 영어 회화 예제

In [57]:
template = """
당신은 영어를 아주 쉽게 가르치는 친절한 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해주세요.
양식은 [FORMAT]를 참고해 주세요.

# 상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

In [58]:
prompt = PromptTemplate.from_template(template)

In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.1, 
    google_api_key=gemini_api_key
)

In [60]:
output_parser = StrOutputParser()

In [61]:
chain = prompt | llm | output_parser

In [62]:
result = chain.invoke({"question": "저는 식당에서 음식을 주문하고 싶어요."})

In [63]:
print(result)

안녕하세요! 식당에서 주문하는 상황, 제가 아주 쉽게 알려드릴게요.
이 대화만 잘 익히면 자신감 있게 주문할 수 있을 거예요! 😊

---

**상황:** 식당에서 음식을 주문하고 싶어요.

**등장인물:**
*   **You (손님):** 당신
*   **Waiter (직원):** 식당 직원

---

- 영어 회화:
**You:** Excuse me!
**Waiter:** Hello! Are you ready to order?
**You:** Yes, I am! What do you recommend?
**Waiter:** Our 'Special Pasta' is very popular. It's delicious!
**You:** Sounds good! I'll have the Special Pasta, please.
**Waiter:** Excellent choice! And to drink?
**You:** I'll have a glass of water, please.
**Waiter:** So, that's one Special Pasta and one glass of water. Is that right?
**You:** Yes, that's perfect. Thank you!
**Waiter:** Great! It will be ready soon.

- 한글 해석:
**You:** 저기요! (직원을 부를 때 쓰는 표현)
**Waiter:** 안녕하세요! 주문하시겠어요?
**You:** 네, 준비됐어요! 어떤 걸 추천하시나요?
**Waiter:** 저희 '스페셜 파스타'가 아주 인기가 많아요. 정말 맛있어요!
**You:** 좋아요! 스페셜 파스타로 할게요.
**Waiter:** 훌륭한 선택이세요! 음료는요?
**You:** 물 한 잔 주세요.
**Waiter:** 그럼, 스페셜 파스타 하나랑 물 한 잔 맞으시죠?
**You:** 네, 맞아요. 감사합니다!
**Waiter:** 알겠습니다! 곧 준비해 드릴게요.

---

**✨ 쉬운 영어 팁!

### JsonOutputPaser

In [64]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

In [65]:
parser

JsonOutputParser()

In [66]:
parser.get_format_instructions()

'Return a JSON object.'

In [67]:
template = """
당신은 영어를 아주 쉽게 가르치는 친절한 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해주세요.
양식은 [FORMAT]를 참고해 주세요.

# 상황:
{question}

#FORMAT:
{format_instructions}
"""

In [71]:
prompt = PromptTemplate(
    template=template,
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [72]:
chain = prompt | llm | parser
result = chain.invoke({"question": "저는 식당에서 음식을 주문하고 싶어요."})

In [74]:
import pprint
pprint.pprint(result)

{'description': '식당에서 음식을 주문할 때 사용할 수 있는 아주 쉽고 친절한 영어 표현들이에요. 자신감을 가지고 주문해 '
                '보세요! 😊',
 'dialogue': [{'line': 'Hello! Welcome! Are you ready to order?',
               'speaker': 'Waiter'},
              {'line': "Yes, I am. I'd like the [dish name], please.",
               'speaker': 'Customer'},
              {'line': 'Excellent choice! And anything to drink with that?',
               'speaker': 'Waiter'},
              {'line': 'Yes, a [drink name], please.', 'speaker': 'Customer'},
              {'line': 'Alright. So, one [dish name] and one [drink name]. Is '
                       'that everything for now?',
               'speaker': 'Waiter'},
              {'line': "Yes, that's all for now, thank you.",
               'speaker': 'Customer'},
              {'line': 'Perfect! Your order will be right out.',
               'speaker': 'Waiter'}],
 'tips': [{'content': "이 두 표현은 식당에서 주문할 때 가장 공손하고 흔하게 쓰이는 표현이에요. 예를 들어, 'I'd "
                      "like the pasta, p

### 일관성 있는 형식으로 출력하기 (JsonOutputParser + Pydantic)


In [75]:
from pydantic import BaseModel, Field

# 원하는 JSON 구조를 Pydantic 클래스로 정의
class EnglishConversation(BaseModel):
    title: str = Field(description="상황에 대한 제목")
    dialogue: list = Field(description="상황에 맞는 영어 대화 리스트")
    explanation: str = Field(description="대화에 대한 쉬운 설명")

In [76]:
# JsonOutputParser에 Pydantic 스키마 전달
parser = JsonOutputParser(pydantic_object=EnglishConversation)

In [78]:
# 프롬프트에 스키마 정보 포함
template = """
당신은 영어를 아주 쉽게 가르치는 친절한 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
출력 양식은 아래 [FORMAT]을 참고해 주세요. 모든 키 이름은 정확히 이 양식과 일치해야 합니다.

# 상황:
{question}

# FORMAT:
{format_instructions}

# 예시:
{{
  "title": "저녁 식사 정하기 (Deciding Dinner)",
  "dialogue": [
    {{"speaker": "A", "text": "What should we have for dinner?"}},
    {{"speaker": "B", "text": "I'm craving some pizza."}}
  ],
  "explanation": "친구와 저녁 메뉴를 정하는 상황입니다. 간단한 표현을 사용했어요."
}}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

In [80]:
# 체인 구성 및 실행
chain = prompt | llm | parser

In [86]:
result = chain.invoke({"question": "식당에서 주문하는 상황"})

In [87]:
pprint.pprint(result)

{'dialogue': [{'speaker': 'Waiter',
               'text': 'Hi there! Welcome to our restaurant. What can I get '
                       'for you today?'},
              {'speaker': 'Customer',
               'text': "Hello! I'd like to order, please."},
              {'speaker': 'Waiter',
               'text': 'Certainly! Have you had a chance to look at our menu?'},
              {'speaker': 'Customer',
               'text': "Yes, I have. I'll have the spaghetti carbonara, "
                       'please.'},
              {'speaker': 'Waiter',
               'text': 'Excellent choice! And would you like anything to drink '
                       'with that?'},
              {'speaker': 'Customer',
               'text': 'Yes, a glass of orange juice, please.'},
              {'speaker': 'Waiter',
               'text': "Alright, so that's one spaghetti carbonara and one "
                       'orange juice. Is there anything else?'},
              {'speaker': 'Customer',
       